In [2]:
from typing import TypedDict

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.graph import END, START, StateGraph
from langgraph.types import Command, interrupt


class ReviewState(TypedDict):
    generated_text: str


def review_node(state: ReviewState) -> ReviewState:
    """审批节点"""
    update = interrupt({"instruction": "请审批并编辑此内容", "content": state["generated_text"]})
    return {"generated_text": update}


builder = StateGraph(ReviewState)
builder.add_node("review", review_node)
builder.add_edge(START, "review")
builder.add_edge("review", END)

graph = builder.compile(InMemorySaver())

config = {"configurable": {"thread_id": "test_view"}}

inital = graph.invoke({"generated_text": "初始草稿"}, config=config)
print(inital.get("__interrupt__", {}))
final_state = graph.invoke(Command(resume=input("修改:")), config=config)
print(final_state)

[Interrupt(value={'instruction': '请审批并编辑此内容', 'content': '初始草稿'}, id='7c664661ed7c7e35f831c9d66f602f49')]
{'generated_text': '不同意'}
